# Skin Lesion Classification with CNN
### 204466 Deep Learning — Final Project
**Dataset:** HAM10000 (Human Against Machine with 10000 training images)  
**Task:** Multi-class skin lesion classification (7 classes)  
**Architecture:** Custom CNN with class imbalance handling

## 1. Setup & Dataset Download

In [ ]:
# Install dependencies
!pip install kaggle -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/skin_lesion/'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

In [ ]:
import os

# Set Kaggle credentials (replace with your own)
os.environ["KAGGLE_USERNAME"] = "your_kaggle_username"   # <-- แก้ตรงนี้
os.environ["KAGGLE_KEY"] = "your_kaggle_api_key"         # <-- แก้ตรงนี้

# Download HAM10000 dataset
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/ham10000

import zipfile
with zipfile.ZipFile("/content/ham10000/skin-cancer-mnist-ham10000.zip", "r") as z:
    z.extractall("/content/ham10000")

print("Dataset downloaded and extracted successfully!")

## 2. Import Libraries

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 3. Data Exploration & Preparation

In [ ]:
# Load metadata
metadata = pd.read_csv("/content/ham10000/HAM10000_metadata.csv")
print(f"Total samples: {len(metadata)}")
print(f"\nColumns: {metadata.columns.tolist()}")
metadata.head()

In [ ]:
# Class label mapping
label_map = {
    'nv':   0,   # Melanocytic nevi
    'mel':  1,   # Melanoma
    'bkl':  2,   # Benign keratosis
    'bcc':  3,   # Basal cell carcinoma
    'akiec':4,   # Actinic keratoses
    'vasc': 5,   # Vascular lesions
    'df':   6    # Dermatofibroma
}
class_names = list(label_map.keys())

metadata['label'] = metadata['dx'].map(label_map)

# Class distribution
plt.figure(figsize=(10, 4))
metadata['dx'].value_counts().plot(kind='bar', color='steelblue')
plt.title('Class Distribution in HAM10000')
plt.xlabel('Lesion Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()
print(metadata['dx'].value_counts())

In [ ]:
# Build image path mapping (images are split across two folders)
image_paths = {}
for folder in ["/content/ham10000/HAM10000_images_part_1",
               "/content/ham10000/HAM10000_images_part_2"]:
    for img_path in glob.glob(os.path.join(folder, "*.jpg")):
        img_id = os.path.splitext(os.path.basename(img_path))[0]
        image_paths[img_id] = img_path

metadata['path'] = metadata['image_id'].map(image_paths)
metadata = metadata.dropna(subset=['path'])
print(f"Images found: {len(metadata)}")

In [ ]:
# Show sample images from each class
fig, axes = plt.subplots(1, 7, figsize=(18, 3))
for i, (cls, label) in enumerate(label_map.items()):
    sample = metadata[metadata['dx'] == cls].iloc[0]
    img = Image.open(sample['path'])
    axes[i].imshow(img)
    axes[i].set_title(cls, fontsize=9)
    axes[i].axis('off')
plt.suptitle('Sample Images per Class')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150)
plt.show()

In [ ]:
# Train / Validation / Test split (70/15/15)
train_df, temp_df = train_test_split(metadata, test_size=0.30, stratify=metadata['label'], random_state=42)
val_df, test_df   = train_test_split(temp_df,  test_size=0.50, stratify=temp_df['label'],  random_state=42)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## 4. Dataset & DataLoader

In [ ]:
class SkinLesionDataset(Dataset):
    """Custom Dataset for HAM10000 skin lesion images."""

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        label = int(row['label'])
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32

# Augmentation for training to reduce overfitting on minority classes
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.763, 0.546, 0.570], std=[0.141, 0.152, 0.169])  # HAM10000 stats
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.763, 0.546, 0.570], std=[0.141, 0.152, 0.169])
])

train_dataset = SkinLesionDataset(train_df, transform=train_transform)
val_dataset   = SkinLesionDataset(val_df,   transform=val_transform)
test_dataset  = SkinLesionDataset(test_df,  transform=val_transform)

# --- Class Imbalance Handling: WeightedRandomSampler ---
# Compute per-class weights: minority classes get sampled more frequently
class_counts = train_df['label'].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for label in train_df['label']]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

## 5. CNN Model Architecture

Custom CNN designed for skin lesion classification:
- 4 convolutional blocks (Conv → BN → ReLU → MaxPool → Dropout)
- Filters: 32 → 64 → 128 → 256
- Global Average Pooling to reduce parameters
- Fully connected classifier: 256 → 128 → 7

In [ ]:
class ConvBlock(nn.Module):
    """Conv2d → BatchNorm → ReLU → MaxPool → Dropout block."""

    def __init__(self, in_channels, out_channels, dropout=0.25):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(dropout)
        )

    def forward(self, x):
        return self.block(x)


class SkinLesionCNN(nn.Module):
    """
    Custom CNN for skin lesion classification.

    Input:  (B, 3, 128, 128)
    Output: (B, 7)  — 7 lesion classes

    Architecture:
        ConvBlock(3→32)    → 64x64
        ConvBlock(32→64)   → 32x32
        ConvBlock(64→128)  → 16x16
        ConvBlock(128→256) →  8x8
        GlobalAvgPool      →  256
        FC(256→128) + ReLU + Dropout
        FC(128→7)
    """

    def __init__(self, num_classes=7):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32,  dropout=0.25),
            ConvBlock(32,  64,  dropout=0.25),
            ConvBlock(64,  128, dropout=0.25),
            ConvBlock(128, 256, dropout=0.40),
        )
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_avg_pool(x)
        x = self.classifier(x)
        return x


model = SkinLesionCNN(num_classes=7).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model: SkinLesionCNN")
print(f"Total parameters: {total_params:,}")
print(model)

## 6. Loss Function, Optimizer & Scheduler

In [ ]:
# Weighted CrossEntropyLoss — penalizes misclassifying rare classes more
weights = torch.tensor(class_weights / class_weights.sum() * len(class_weights), dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Reduce LR when validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

print("Loss: CrossEntropyLoss (class-weighted)")
print("Optimizer: Adam (lr=1e-3, weight_decay=1e-4)")
print("Scheduler: ReduceLROnPlateau (patience=3, factor=0.5)")

## 7. Training Loop

In [ ]:
NUM_EPOCHS = 30
best_val_loss = float('inf')

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)

    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), SAVE_DIR + 'best_model.pth')

    print(f"Epoch [{epoch:02d}/{NUM_EPOCHS}]  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}")

print(f"\nTraining complete. Best model saved to {SAVE_DIR}best_model.pth")

In [ ]:
NUM_EPOCHS = 30
best_val_loss = float('inf')

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)

    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')

    print(f"Epoch [{epoch:02d}/{NUM_EPOCHS}]  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}")

print("\nTraining complete. Best model saved to best_model.pth")

epochs = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs, history['train_loss'], label='Train Loss', color='royalblue')
ax1.plot(epochs, history['val_loss'],   label='Val Loss',   color='tomato')
ax1.set_title('Loss Curve')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, history['train_acc'], label='Train Acc', color='royalblue')
ax2.plot(epochs, history['val_acc'],   label='Val Acc',   color='tomato')
ax2.set_title('Accuracy Curve')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(SAVE_DIR + 'training_curves.png', dpi=150)
plt.show()

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs, history['train_loss'], label='Train Loss', color='royalblue')
ax1.plot(epochs, history['val_loss'],   label='Val Loss',   color='tomato')
ax1.set_title('Loss Curve')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, history['train_acc'], label='Train Acc', color='royalblue')
ax2.plot(epochs, history['val_acc'],   label='Val Acc',   color='tomato')
ax2.set_title('Accuracy Curve')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

model.load_state_dict(torch.load(SAVE_DIR + 'best_model.pth', map_location=device))

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('best_model.pth', map_location=device))

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Test Set')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig(SAVE_DIR + 'confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Test Set')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 10. Visualize Predictions

In [ ]:
# Show 8 test samples with predicted vs actual label
model.eval()
fig, axes = plt.subplots(2, 4, figsize=(15, 7))

inv_normalize = transforms.Normalize(
    mean=[-0.763/0.141, -0.546/0.152, -0.570/0.169],
    std=[1/0.141, 1/0.152, 1/0.169]
)

sample_images, sample_labels = next(iter(test_loader))
sample_images_dev = sample_images.to(device)

with torch.no_grad():
    outputs = model(sample_images_dev)
    probs = F.softmax(outputs, dim=1)
    _, preds = outputs.max(1)

for i, ax in enumerate(axes.flat):
    img = inv_normalize(sample_images[i]).permute(1, 2, 0).clamp(0, 1).numpy()
    actual = class_names[sample_labels[i]]
    predicted = class_names[preds[i].item()]
    confidence = probs[i][preds[i]].item()
    color = 'green' if actual == predicted else 'red'
    ax.imshow(img)
    ax.set_title(f"Pred: {predicted}\nActual: {actual}\n({confidence:.1%})",
                 color=color, fontsize=9)
    ax.axis('off')

plt.suptitle('Predictions (green=correct, red=wrong)', fontsize=12)
plt.tight_layout()
plt.savefig('predictions.png', dpi=150)
plt.show()

import json
from sklearn.metrics import precision_score, recall_score, f1_score

results = {
    "model": "Custom CNN (SkinLesionCNN)",
    "accuracy": float(test_acc),
    "precision": float(precision_score(all_labels, all_preds, average='weighted', zero_division=0)),
    "recall":    float(recall_score(all_labels, all_preds, average='weighted', zero_division=0)),
    "f1":        float(f1_score(all_labels, all_preds, average='weighted', zero_division=0)),
    "per_class_f1": {
        class_names[i]: float(f)
        for i, f in enumerate(f1_score(all_labels, all_preds, average=None, zero_division=0))
    }
}

with open(SAVE_DIR + 'cnn_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {SAVE_DIR}cnn_results.json")
print(f"  Accuracy : {results['accuracy']:.4f}")
print(f"  Precision: {results['precision']:.4f}")
print(f"  Recall   : {results['recall']:.4f}")
print(f"  F1       : {results['f1']:.4f}")

In [ ]:
import json
from sklearn.metrics import precision_score, recall_score, f1_score

results = {
    "model": "Custom CNN (SkinLesionCNN)",
    "accuracy": float(test_acc),
    "precision": float(precision_score(all_labels, all_preds, average='weighted', zero_division=0)),
    "recall":    float(recall_score(all_labels, all_preds, average='weighted', zero_division=0)),
    "f1":        float(f1_score(all_labels, all_preds, average='weighted', zero_division=0)),
    "per_class_f1": {
        class_names[i]: float(f)
        for i, f in enumerate(f1_score(all_labels, all_preds, average=None, zero_division=0))
    }
}

with open('cnn_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to cnn_results.json")
print(f"  Accuracy : {results['accuracy']:.4f}")
print(f"  Precision: {results['precision']:.4f}")
print(f"  Recall   : {results['recall']:.4f}")
print(f"  F1       : {results['f1']:.4f}")

## Summary

| Component | Detail |
|-----------|--------|
| **Dataset** | HAM10000 — 10,015 dermoscopy images, 7 classes |
| **Architecture** | Custom CNN: 4 ConvBlocks + Global Avg Pool + FC |
| **Imbalance handling** | WeightedRandomSampler + Weighted CrossEntropyLoss |
| **Augmentation** | Flip, Rotate, ColorJitter |
| **Optimizer** | Adam (lr=1e-3, wd=1e-4) |
| **Scheduler** | ReduceLROnPlateau |
| **Epochs** | 30 |
| **Input size** | 128×128 RGB |